In [ ]:
from pathlib import Path

import AERzip


def find_compressed_dir() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidate = base / "Compressed files"
        if candidate.exists():
            return candidate
    raise FileNotFoundError("Could not find the 'Compressed files' folder from the current notebook location.")


compressed_dir = find_compressed_dir()
compressed_files = sorted(
    [
        p for p in compressed_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in {".aedat", ".gz", ".zip"}
    ]
)

failed_files = []

for file_path in compressed_files:
    try:
        addresses_test, timestamps_test = AERzip.loadCompressedFile(str(file_path), verbose=False)
        print(f"OK   {file_path} -> {len(addresses_test)} events")
    except Exception as exc:
        failed_files.append((file_path, exc))
        print(f"FAIL {file_path} -> {exc}")

print()
print(f"Checked {len(compressed_files)} files.")
print(f"Successful: {len(compressed_files) - len(failed_files)}")
print(f"Failed: {len(failed_files)}")

if failed_files:
    print("\nFailed files:")
    for file_path, exc in failed_files:
        print(f"- {file_path}: {exc}")